## AGRÉGATION SILVER → GOLD

In [ ]:
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

#LORSQU'ON TRAVAILLE DEPUIS SA MACHINE LOCAL
MINIO_ENDPOINT = "http://192.168.1.230:30137"
MINIO_ACCESS_KEY = "datalab-team"
MINIO_SECRET_KEY = "minio-datalabteam123"
NESSIE_URI = "http://192.168.1.230:30604/api/v1"

#---------------------------------------------------------------------------------

#LORSQU'ON TRAVAILLE SUR JHUB
# MINIO_ENDPOINT = "http://minio.mon-namespace.svc.cluster.local:80"
# MINIO_ACCESS_KEY = "datalab-team"
# MINIO_SECRET_KEY = "minio-datalabteam123"
# NESSIE_URI = "http://nessie.trino.svc.cluster.local:19120/api/v1"

#---------------------------------------------------------------------------------

conf = (
    SparkConf()
    .setAppName("Silver_to_Gold")
    .set("spark.driver.host", "127.0.0.1")
    .set("spark.driver.bindAddress", "127.0.0.1")
    .set("spark.driver.memory", "16g")
    .set("spark.jars.packages",
         "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1,"
         "org.apache.hadoop:hadoop-aws:3.3.4,"
         "org.projectnessie.nessie-integrations:nessie-spark-extensions-3.5_2.12:0.77.1")

    .set("spark.sql.extensions",
         "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
         "org.projectnessie.spark.extensions.NessieSparkSessionExtensions")

    .set("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .set("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .set("spark.sql.catalog.nessie.uri", NESSIE_URI)
    .set("spark.sql.catalog.nessie.ref", "main")
    .set("spark.sql.catalog.nessie.warehouse", "s3a://bronze/")

    .set("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .set("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .set("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
    .set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
)

In [ ]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()

In [ ]:
# Lecture de la (ou des) table(s) Silver
df = spark.table("nessie.silver.<<nom_de_la_table>>")
print(f"Silver : {df.count():,} lignes")

## ON EFFECTUE LES AGRÉGATIONS GOLD ICI

Gold = tables prêtes pour les dashboards et les exports finaux.  
Elles sont dénormalisées, pré-agrégées, optimisées pour la lecture.

In [ ]:
# --- OPTION A : Agrégation DataFrame API ---

df_gold = (
    df
    .groupBy("<<dimension_1>>", "<<dimension_2>>")
    .agg(
        F.count("*").alias("nb"),
        F.sum("<<montant>>").alias("total"),
        F.avg("<<montant>>").alias("moyenne"),
        F.percentile_approx("<<montant>>", 0.5).alias("mediane"),
    )
    .orderBy("<<dimension_1>>", "<<dimension_2>>")
)

In [ ]:
# --- OPTION B : Agrégation SQL pur ---

# df.createOrReplaceTempView("silver_source")
#
# df_gold = spark.sql("""
#     SELECT
#         <<dimension_1>>,
#         <<dimension_2>>,
#         COUNT(*)                                  AS nb,
#         SUM(<<montant>>)                          AS total,
#         AVG(<<montant>>)                          AS moyenne,
#         PERCENTILE_APPROX(<<montant>>, 0.5)       AS mediane
#     FROM silver_source
#     GROUP BY <<dimension_1>>, <<dimension_2>>
#     ORDER BY <<dimension_1>>, <<dimension_2>>
# """)

# --- JOINTURE MULTI-TABLES (ex. Silver × référentiel) ---
# df_ref = spark.table("nessie.silver.referentiel")
# df_gold = df.join(df_ref, on="id_commun", how="left")

In [ ]:
print(f"Gold : {df_gold.count():,} lignes × {len(df_gold.columns)} colonnes")
df_gold.show(10)

In [ ]:
TABLE_GOLD = "nessie.gold.<<nom_du_resultat>>"

spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.gold")

df_gold.write \
    .format("iceberg") \
    .mode("overwrite") \
    .saveAsTable(TABLE_GOLD)

In [ ]:
# Export CSV vers staging pour partage externe (dashboard, équipes sans Trino)
# df_gold.coalesce(1) \
#     .write \
#     .option("header", "true") \
#     .option("delimiter", ";") \
#     .mode("overwrite") \
#     .csv("s3a://staging/exports/<<nom_du_resultat>>/")